# Yuda 6 — Final Improvements
Better prompts, Focal Loss fix, 3-model ensemble, TTA

In [1]:
import sys
sys.path.insert(0, "../..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import gc
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader

import open_clip
import timm

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.utils.load_data import load_train
from modules.models.factory import TrashClassifier
from modules.data.dataset import get_dataloaders, get_transforms

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"open_clip: {open_clip.__version__}")

/home/prayudahlah/projects/external/big-data-big-trouble/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
open_clip: 3.3.0


In [2]:
_CLASS_TO_IDX = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}

CLIP_NAME = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'
clip_model, _, clip_transform = open_clip.create_model_and_transforms(
    CLIP_NAME, pretrained=CLIP_PRETRAINED
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_NAME)
clip_model = clip_model.to('cpu')
clip_model.eval()
clip_visual = clip_model.visual.to(DEVICE)
clip_dim = clip_visual.output_dim
print(f"CLIP dim: {clip_dim}")

conv_val_tfm = get_transforms(augment=False, img_size=224)
conv_val_tfm_300 = get_transforms(augment=False, img_size=300)

results = []

CLIP dim: 512


In [3]:
train_loader, val_loader, val_ds = get_dataloaders(batch_size=32, num_workers=4)
df_train = train_loader.dataset.df
df_val = val_loader.dataset.df
print(f"Train: {len(df_train)} | Val: {len(df_val)}")

class SingleTransformDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), _CLASS_TO_IDX[row['label']]

clip_train_ds = SingleTransformDataset(df_train, clip_transform)
clip_val_ds = SingleTransformDataset(df_val, clip_transform)
clip_train_loader = DataLoader(clip_train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
clip_val_loader = DataLoader(clip_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

conv_val_ds = SingleTransformDataset(df_val, conv_val_tfm)
conv_val_loader = DataLoader(conv_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

conv_val_ds_300 = SingleTransformDataset(df_val, conv_val_tfm_300)
conv_val_loader_300 = DataLoader(conv_val_ds_300, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

Train: 21221 | Val: 5306


---
## Setup: Model, Loss, Trainer

In [4]:
class CLIPWithTextFeatures(nn.Module):
    def __init__(self, clip_model, clip_visual, prompts, num_classes=3):
        super().__init__()
        self.visual = clip_visual
        self.encoder = self.visual
        self.logit_scale = clip_model.logit_scale
        dim = self.visual.output_dim
        text_tokens = clip_tokenizer(prompts)
        with torch.no_grad():
            text_feat = clip_model.encode_text(text_tokens)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        self.register_buffer('text_features', text_feat)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(dim + 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        v = self.visual(x)
        v_norm = v / v.norm(dim=-1, keepdim=True)
        sim = v_norm @ self.text_features.T
        combined = torch.cat([v, sim * self.logit_scale.exp()], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = True


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        log_sm = F.log_softmax(inputs, dim=1)
        log_pt = log_sm[range(len(targets)), targets]
        pt = log_pt.exp()
        focal = -((1 - pt).clamp(min=1e-8).pow(self.gamma)) * log_pt
        if self.alpha is not None:
            focal = self.alpha[targets] * focal
        return focal.mean()


def train_head_only(model, train_loader, val_loader, name, epochs=10, criterion=None):
    model = model.to(DEVICE)
    model.freeze_encoder()
    criterion = criterion or nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        for x, y in tqdm(train_loader, desc=f"  E{epoch+1}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
        sched.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds.extend(model(x).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
        print(f"  E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")
    model.load_state_dict(best_state)
    result = {'name': name, 'best_val_f1': best_f1}
    torch.save(best_state, RESULTS / f'{name}.pt')
    json.dump(result, open(RESULTS / f'{name}.json', 'w'))
    return model, best_f1

---
## Exp 1: Better Prompts + CE Loss

In [5]:
print("=" * 50)
print("Exp 1: Better Prompts + CE")
print("=" * 50)

prompts_improved = [
    "a photo of recyclable items like paper, plastic, glass, and metal",
    "a photo of electronic waste like computers, phones, cables, and batteries",
    "a photo of organic waste like food scraps, leaves, and plants",
]

m1 = CLIPWithTextFeatures(clip_model, clip_visual, prompts_improved)
m1, f1_1 = train_head_only(m1, clip_train_loader, clip_val_loader, 'yuda6_prompt_ce', epochs=10)
print(f"Exp 1 (better prompts + CE): {f1_1:.4f}")
results.append({'name': '1_prompt_ce', 'best_val_f1': f1_1})

Exp 1: Better Prompts + CE


  E1: val_f1=0.9697 (best=0.9697)


  E2: val_f1=0.9769 (best=0.9769)


  E3: val_f1=0.9770 (best=0.9770)


  E4: val_f1=0.9792 (best=0.9792)


...

  E6: val_f1=0.9813 (best=0.9813)


  E7: val_f1=0.9817 (best=0.9817)


  E8: val_f1=0.9825 (best=0.9825)


  E9: val_f1=0.9811 (best=0.9825)


  E10: val_f1=0.9816 (best=0.9825)
Exp 1 (better prompts + CE): 0.9825


## Exp 2: Better Prompts + Focal Loss (Fixed)

In [6]:
print("=" * 50)
print("Exp 2: Better Prompts + Focal Loss")
print("=" * 50)

# class weights: inverse frequency
from modules.data.dataset import TrashDataset
temp_ds = TrashDataset(df_train)
counts = np.bincount([_CLASS_TO_IDX[l] for l in df_train['label']], minlength=3)
class_weights = torch.FloatTensor(counts.sum() / (3 * counts)).to(DEVICE)
focal = FocalLoss(alpha=class_weights, gamma=2.0)
print(f"Class weights: {class_weights}")

m2 = CLIPWithTextFeatures(clip_model, clip_visual, prompts_improved)
m2, f1_2 = train_head_only(m2, clip_train_loader, clip_val_loader, 'yuda6_prompt_focal', epochs=10, criterion=focal)
print(f"Exp 2 (better prompts + Focal): {f1_2:.4f}")
results.append({'name': '2_prompt_focal', 'best_val_f1': f1_2})

Exp 2: Better Prompts + Focal Loss
Class weights: tensor([0.8843, 2.2321, 0.7036], device='cuda:0')


  E1: val_f1=0.9696 (best=0.9696)


  E2: val_f1=0.9758 (best=0.9758)


  E3: val_f1=0.9772 (best=0.9772)


  E4: val_f1=0.9798 (best=0.9798)


...

  E6: val_f1=0.9825 (best=0.9825)


  E7: val_f1=0.9816 (best=0.9825)


  E8: val_f1=0.9831 (best=0.9831)


  E9: val_f1=0.9832 (best=0.9832)


  E10: val_f1=0.9839 (best=0.9839)
Exp 2 (better prompts + Focal): 0.9839


---
## Ensemble Prep
Generate probabilities from all models on validation set

In [7]:
def get_probs(model, loader, use_tta=False):
    """Run model on loader, return softmax probabilities [N, 3]"""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc="Inference", leave=False):
            x = x.to(DEVICE)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            if use_tta:
                flipped = torch.flip(x, dims=[3])
                probs += torch.softmax(model(flipped), dim=1)
                probs /= 2
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs)


val_labels = np.array([_CLASS_TO_IDX[r['label']] for _, r in df_val.iterrows()])
print(f"Val labels: {len(val_labels)}")

Val labels: 5306


In [8]:
# Pick best C model
if f1_1 >= f1_2:
    best_c = m1
    best_c_name = 'prompt_ce'
    best_c_f1 = f1_1
else:
    best_c = m2
    best_c_name = 'prompt_focal'
    best_c_f1 = f1_2
print(f"Best C model: {best_c_name} @ {best_c_f1:.4f}")

print("Generating C probs...")
c_probs = get_probs(best_c, clip_val_loader)
del best_c, m1, m2
torch.cuda.empty_cache()

print("Generating ConvNeXt probs...")
convnext = TrashClassifier('convnext_tiny', num_classes=3).to(DEVICE)
convnext.load_state_dict(torch.load(RESULTS / 'yuda_convnext_tiny.pt', map_location=DEVICE, weights_only=True))
convnext.eval()
conv_probs = get_probs(convnext, conv_val_loader)
del convnext
torch.cuda.empty_cache()

print("Generating EfficientNet-B3 @300 probs...")
effnet = TrashClassifier('efficientnet_b3', num_classes=3).to(DEVICE)
effnet.load_state_dict(torch.load(RESULTS / 'yuda2_effnet_b3_300.pt', map_location=DEVICE, weights_only=True))
effnet.eval()
eff_probs = get_probs(effnet, conv_val_loader_300)
del effnet
torch.cuda.empty_cache()

print(f"C: {c_probs.shape} | ConvNeXt: {conv_probs.shape} | EffNet: {eff_probs.shape}")

Best C model: prompt_focal @ 0.9839
Generating C probs...


Generating ConvNeXt probs...


Generating EfficientNet-B3 @300 probs...


C: (5306, 3) | ConvNeXt: (5306, 3) | EffNet: (5306, 3)


---
## Exp 3: Grid Search Ensemble Weights + TTA

In [9]:
print("=" * 50)
print("Exp 3: Ensemble Weight Search + TTA")
print("=" * 50)

best_row = None
rows = []

# 2-model: C + ConvNeXt
for w_c in np.arange(0.50, 0.91, 0.05):
    w_conv = 1.0 - w_c
    ensemble = w_c * c_probs + w_conv * conv_probs
    preds = ensemble.argmax(axis=1)
    f1 = f1_score(val_labels, preds, average='macro')
    rows.append({'name': f'C{w_c:.2f}+Conv{w_conv:.2f}', 'best_val_f1': f1})

# 3-model: C + ConvNeXt + EffNet
for w_c in [0.5, 0.6, 0.7, 0.8]:
    remainder = 1.0 - w_c
    for w_conv in [0.1, 0.2, 0.3, 0.4]:
        w_eff = remainder - w_conv
        if w_eff < 0.05:
            continue
        ensemble = w_c * c_probs + w_conv * conv_probs + w_eff * eff_probs
        preds = ensemble.argmax(axis=1)
        f1 = f1_score(val_labels, preds, average='macro')
        rows.append({'name': f'C{w_c:.2f}+Conv{w_conv:.2f}+Eff{w_eff:.2f}', 'best_val_f1': f1})

# TTA on best 2-model weights
print("Running TTA...")
best_c_tta = CLIPWithTextFeatures(clip_model, clip_visual, prompts_improved).to(DEVICE)
best_c_tta.load_state_dict(torch.load(RESULTS / f'yuda6_prompt_ce.pt', map_location=DEVICE, weights_only=True))
best_c_tta.eval()
c_probs_tta = get_probs(best_c_tta, clip_val_loader, use_tta=True)
del best_c_tta
torch.cuda.empty_cache()

convnext_tta = TrashClassifier('convnext_tiny', num_classes=3).to(DEVICE)
convnext_tta.load_state_dict(torch.load(RESULTS / 'yuda_convnext_tiny.pt', map_location=DEVICE, weights_only=True))
convnext_tta.eval()
conv_probs_tta = get_probs(convnext_tta, conv_val_loader, use_tta=True)
del convnext_tta
torch.cuda.empty_cache()

for w_c in np.arange(0.50, 0.91, 0.05):
    w_conv = 1.0 - w_c
    ensemble = w_c * c_probs_tta + w_conv * conv_probs_tta
    preds = ensemble.argmax(axis=1)
    f1 = f1_score(val_labels, preds, average='macro')
    rows.append({'name': f'TTA_C{w_c:.2f}+Conv{w_conv:.2f}', 'best_val_f1': f1})

df_rows = pd.DataFrame(rows)
best_row = df_rows.loc[df_rows['best_val_f1'].idxmax()]
print(f"\nBest: {best_row['name']} @ {best_row['best_val_f1']:.4f}")
results.extend(rows)

Exp 3: Ensemble Weight Search + TTA
Running TTA...



Best: C0.70+Conv0.10+Eff0.20 @ 0.9856


---
## Summary

In [10]:
summary = pd.DataFrame(results)
summary = summary.sort_values('best_val_f1', ascending=False)
summary.to_csv(RESULTS / 'yuda6_summary.csv', index=False)
print("Top 10:")
print(summary.head(10).to_string())

best = summary.iloc[0]
print(f"\n🏆 Champion: {best['name']} @ {best['best_val_f1']:.4f}")

Top 10:
                      name  best_val_f1
18  C0.70+Conv0.10+Eff0.20     0.985616
20  C0.80+Conv0.10+Eff0.10     0.984873
15  C0.60+Conv0.10+Eff0.30     0.984695
8           C0.80+Conv0.20     0.984420
16  C0.60+Conv0.20+Eff0.20     0.984398
7           C0.75+Conv0.25     0.984273
9           C0.85+Conv0.15     0.983971
26      TTA_C0.75+Conv0.25     0.983953
1           2_prompt_focal     0.983950
19  C0.70+Conv0.20+Eff0.10     0.983676

🏆 Champion: C0.70+Conv0.10+Eff0.20 @ 0.9856
